### `etd_rk2.ipyng`  
*Created June 12, 2026*

**Description:** This notebook implements the second-order ETD-RK2 method, an exponential time differencing (ETD) integrator for ODE systems of the form 
$$ \frac{du}{dt} = Au + F(u,t) $$ 

where $A$ is an $N \times N$ matrix and $F$ is a non-linear function from $\mathbb{R}^N \times \mathbb{R}$ to $\mathbb{R}^N$. The ETD-RK2 algorithm is as follows: 
\begin{align*}
    a_n &= u_n e^{\Delta t A} + \Delta t \,\big[\varphi_1(\Delta t A) F(u_n,t_n) \big]  \\[5pt]
   u_{n+1} &= a_n + \Delta t \, \varphi_2(\Delta t A) \,\left[F(a_n,t_{n+1}) - F(a_n,t_n) \right]
\end{align*}

where $\varphi_1(\Delta t A)$ and $\varphi_2(\Delta t A)$ are $N \times N$ vectors, and $N(u_n,t_n)$ and $F(a_n,t_n)$ are $N \times 1$ column vectors and 
\begin{align*}
   \varphi_1(z) = \frac{e^z - 1}{z} \qquad \text{ and } \qquad \varphi_2(z) = \frac{e^z - 1 - z}{z^2} \qquad \text{for } z \in \mathbb{C}.
\end{align*}

\begin{align*}
  \scriptstyle \varphi_1\left(\begin{bmatrix} \lambda_1 & & \\ & \ddots & \\ & & \lambda_N \end{bmatrix} \right) = \begin{bmatrix} \varphi_1(\lambda_1) & & \\ & \ddots & \\  & & \varphi_1(\lambda_N) \end{bmatrix} \qquad \text{ and } \qquad \scriptstyle \varphi_2\left(\begin{bmatrix} \lambda_1 & & \\ & \ddots & \\ & & \lambda_N \end{bmatrix} \right) = \begin{bmatrix} \varphi_2(\lambda_1) & & \\ & \ddots & \\  & & \varphi_2(\lambda_N) \end{bmatrix}, \qquad \lambda_1,\ldots,\lambda_N \in \mathbb{C}
\end{align*}


\begin{align*}
   \varphi_1(B) := \sum_{k=0}^{\infty} \frac{B^k}{(k+1)!} \qquad \text{and} \qquad \varphi_2(B) := \sum_{k=0}^{\infty} \frac{B^k}{(k+2)!} \qquad \text{for } B \in \mathbb{C}^{N \times N}. 
\end{align*}

In [1]:
using LinearAlgebra, LaTeXStrings, Random, Printf, NBInclude, Statistics, UnPack
@nbinclude("phi_functions.ipynb")

In [2]:
function etd_rk2(A, f, u0; tspan::Tuple, Δt::Float64, p = nothing, ϵ::Float64 = 1e-3)
    """
    PARAMETERS:
    ----------
    A :: N x N constant matrix
    f :: nonlinear function from R^N to R^N 
    u0 :: initial condition (vector of length N)
    Δt :: step size 
    p :: parameter for the function `f` (if required)  
    """
    
    t0, tf = tspan
    M = floor(Int, (tf - t0) / Δt)                      #Number of time subintervals
    t = collect(range(t0, t0 + M*Δt, length = M + 1))   #Time values at which to record the solution (incl t0)
    u = [zero(u0) for j=1:M+1]                          #Vector (of vectors) to store the solution iterates u_0,…,u_M
    u[1] = u0

    Φ₀, Φ₁, Φ₂ = phis(A*Δt, 2; ϵ = ϵ)
  
    for n=1:M      #Compute u[2],...,u[M+1] 
        a = Φ₀ * u[n] + Δt * Φ₁ * f(u[n], p, t[n])
        u[n+1] = a + Δt * Φ₂ * (f(a, p, t[n+1]) - f(u[n], p, t[n]))
    end 

    return (u = u, t = t, Δt = Δt, p = p) 
end 

etd_rk2 (generic function with 1 method)